# Crawler Dataset Artikel Musik Indonesia dari ANTARA News

Notebook ini dibuat untuk Tugas 1 mata kuliah **Temu Kembali Informasi / Information Retrieval**. Tujuannya adalah mengumpulkan corpus artikel web bertopik **Musik Indonesia** dari website **ANTARA News**. Dataset hasil crawling dapat digunakan kembali pada Tugas 2 untuk preprocessing, inverted index, Boolean Retrieval, TF-IDF / Vector Space Model, cosine similarity, BM25, ranking, dan evaluasi retrieval.

Dataset yang ditargetkan berupa artikel teks yang cukup panjang dan relevan, bukan sekadar daftar judul atau metadata. Crawling dilakukan secara etis, hanya pada halaman publik, mengikuti robots.txt, menggunakan delay antar-request, dan tidak melakukan bypass login, CAPTCHA, paywall, atau rate limit.

## 1. Pendahuluan

**Sumber data:** ANTARA News (`antaranews.com`)

**Topik:** Musik Indonesia

ANTARA dipilih karena merupakan portal berita nasional yang menyediakan artikel publik dengan struktur berita yang relatif konsisten. Artikel-artikel bertopik musik Indonesia cocok digunakan sebagai dokumen corpus untuk eksperimen Information Retrieval karena memiliki judul, URL, tanggal publikasi, dan isi artikel yang dapat diekstrak sebagai teks.

## 2. Import Library

Notebook ini mengutamakan library yang umum tersedia di Google Colab. Jika `beautifulsoup4` belum tersedia, jalankan instalasi berikut.

In [ ]:
# Jalankan hanya jika BeautifulSoup belum tersedia di environment.
# !pip install beautifulsoup4

In [ ]:
import hashlib
import re
import time
from collections import deque
from datetime import datetime
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup


## 3. Konfigurasi Crawler

In [ ]:
BASE_URL = "https://www.antaranews.com"
ROBOTS_URL = "https://www.antaranews.com/robots.txt"

SEED_URL = "https://www.antaranews.com/tag/musik-indonesia"
SEED_URLS = [
    "https://www.antaranews.com/tag/musik-indonesia",
    "https://www.antaranews.com/tag/musisi-indonesia",
    "https://www.antaranews.com/tag/industri-musik",
    "https://www.antaranews.com/tag/musik",
    "https://www.antaranews.com/tag/lagu",
]

TARGET_DOCUMENTS = 1000
MIN_REQUIRED_DOCUMENTS = 100
MAX_PAGES = 350
MAX_ARTICLE_CANDIDATES = 2000
REQUEST_DELAY = 1.5
TIMEOUT = 20
MIN_CONTENT_LENGTH = 500
MUSIC_TERMS = {
    "musik", "musisi", "musikal", "lagu", "penyanyi", "band",
    "grup musik", "album", "single", "singel", "konser", "festival musik",
    "gitar", "drum", "vokal", "rekaman", "rilis", "royalti",
    "komposer", "pencipta lagu", "industri musik", "label musik",
    "streaming musik", "dangdut", "pop", "jazz", "rock", "hip hop",
    "keroncong", "gamelan", "soundtrack", "aransemen", "orkestra",
    "tangga lagu", "produser musik", "produksi musik",
}
STRONG_MUSIC_TERMS = {
    "musik", "musisi", "lagu", "penyanyi", "band", "grup musik",
    "album", "single", "singel", "konser", "festival musik",
    "rekaman", "rilis", "royalti", "komposer", "pencipta lagu",
    "industri musik", "label musik", "streaming musik", "soundtrack",
}
INCIDENTAL_SUBJECT_TERMS = {
    "otomotif", "mobil", "kendaraan", "motor", "byd", "openai",
    "teknologi", "ai", "politik", "pemilu", "pemerintah", "panglima",
    "tni", "presiden", "pariwisata", "wisata", "hotel", "destinasi",
    "sepak bola", "investasi",
    "saham", "ekonomi makro", "kecelakaan",
}
RELEVANCE_MIN_MUSIC_HITS = 3
REJECTED_EXAMPLES_LIMIT = 10
OUTPUT_FILE = "dataset_musik_indonesia_antara.csv"
USER_AGENT = "IR-Course-Crawler/1.0 (educational dataset collection; polite crawling)"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

stats = {
    "listing_pages_requested": 0,
    "article_pages_requested": 0,
    "requests_success": 0,
    "requests_failed": 0,
    "http_403": 0,
    "http_404": 0,
    "http_429": 0,
    "http_5xx": 0,
    "total_urls_found_raw": 0,
    "total_unique_article_urls": 0,
    "articles_extracted": 0,
    "articles_failed": 0,
    "empty_content": 0,
    "duplicates": 0,
    "discarded_cleaning": 0,
    "rate_limit_events": 0,
    "irrelevant_articles": 0,
    "relevant_articles": 0,
}

request_log = []
crawl_warnings = []
rejected_examples = []


## 4. Pemeriksaan Robots.txt dan Etika Crawling

Sebelum crawling, notebook mengambil dan menampilkan `robots.txt`, lalu menggunakan `RobotFileParser` untuk mengecek apakah URL boleh diakses oleh user-agent crawler. Jika ada konflik dengan robots.txt, URL tersebut tidak akan di-crawl.

In [ ]:
def fetch_text(url, timeout=TIMEOUT):
    try:
        response = SESSION.get(url, timeout=timeout)
        return response.status_code, response.text
    except requests.exceptions.RequestException as exc:
        return None, str(exc)

robots_status, robots_text = fetch_text(ROBOTS_URL)
print("Robots URL:", ROBOTS_URL)
print("HTTP status:", robots_status)
print(robots_text[:5000] if robots_text else "robots.txt tidak dapat dibaca")

rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
try:
    rp.read()
    robots_available = True
except Exception as exc:
    robots_available = False
    crawl_warnings.append(f"Gagal membaca robots.txt dengan RobotFileParser: {exc}")

print("\nUser-Agent crawler:", USER_AGENT)
print("Delay antar-request:", REQUEST_DELAY, "detik")
print("Seed boleh di-crawl menurut robots.txt:", rp.can_fetch(USER_AGENT, SEED_URL) if robots_available else False)

In [ ]:
def extract_relevant_robot_rules(robots_text):
    rules = []
    if not robots_text:
        return rules
    for line in robots_text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        lower = stripped.lower()
        if lower.startswith(("user-agent:", "allow:", "disallow:", "crawl-delay:")):
            rules.append(stripped)
    return rules

print("Aturan robots.txt yang relevan:")
for rule in extract_relevant_robot_rules(robots_text)[:200]:
    print(rule)

In [ ]:
# Pemeriksaan Terms of Service / kebijakan penggunaan jika URL umum tersedia.
TOS_CANDIDATES = [
    "https://www.antaranews.com/terms-of-service",
    "https://www.antaranews.com/ketentuan-penggunaan",
    "https://www.antaranews.com/tentang-kami",
]

tos_results = []
for tos_url in TOS_CANDIDATES:
    allowed = rp.can_fetch(USER_AGENT, tos_url) if robots_available else False
    if not allowed:
        tos_results.append({"url": tos_url, "status": "blocked_by_robots_or_unchecked"})
        continue
    time.sleep(REQUEST_DELAY)
    status, text = fetch_text(tos_url)
    tos_results.append({"url": tos_url, "status": status, "preview": (text or "")[:300]})

pd.DataFrame(tos_results)

## 5. Fungsi Helper

Fungsi di bawah memisahkan request halaman, validasi URL, pengecekan robots.txt, ekstraksi link artikel, pagination, ekstraksi artikel, cleaning teks, deduplication, dan crawling.

In [ ]:
def now_indonesia_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def is_antaranews_domain(url):
    netloc = urlparse(url).netloc.lower()
    return netloc in {"antaranews.com", "www.antaranews.com"}

def normalize_url(url, base=BASE_URL):
    if not url:
        return None
    absolute = urljoin(base, url.strip())
    parsed = urlparse(absolute)
    if parsed.scheme not in {"http", "https"}:
        return None
    netloc = parsed.netloc.lower()
    path = re.sub(r"/{2,}", "/", parsed.path).rstrip("/")
    allowed_query_keys = {"page"}
    query_items = [(k, v) for k, v in parse_qsl(parsed.query) if k.lower() in allowed_query_keys]
    query = urlencode(query_items)
    return urlunparse(("https", netloc, path, "", query, ""))

def can_fetch_url(url):
    if not robots_available:
        return False
    try:
        return rp.can_fetch(USER_AGENT, url)
    except Exception:
        return False

def is_blocked_or_irrelevant_path(url):
    path = urlparse(url).path.lower()
    blocked_fragments = [
        "/login", "/register", "/search", "/tag/", "/kategori/", "/video/",
        "/foto/", "/rss", "/ajax", "/api", "/cdn-cgi", "/user", "/komentar"
    ]
    return any(fragment in path for fragment in blocked_fragments)

def looks_like_article_url(url):
    parsed = urlparse(url)
    path = parsed.path.strip("/")
    if not path or is_blocked_or_irrelevant_path(url):
        return False
    # Pola umum artikel ANTARA: /berita/1234567/judul-artikel
    return bool(re.match(r"^berita/\d+/[a-z0-9-]+", path))

def polite_get(url, purpose="page", delay=REQUEST_DELAY):
    if not is_antaranews_domain(url):
        stats["requests_failed"] += 1
        return None
    if not can_fetch_url(url):
        stats["requests_failed"] += 1
        request_log.append({"url": url, "purpose": purpose, "status": "blocked_by_robots", "time": now_indonesia_str()})
        return None
    time.sleep(delay)
    try:
        response = SESSION.get(url, timeout=TIMEOUT)
        status = response.status_code
        request_log.append({"url": url, "purpose": purpose, "status": status, "time": now_indonesia_str()})
        if purpose == "listing":
            stats["listing_pages_requested"] += 1
        elif purpose == "article":
            stats["article_pages_requested"] += 1
        if status == 200:
            stats["requests_success"] += 1
            return response
        stats["requests_failed"] += 1
        if status == 403:
            stats["http_403"] += 1
        elif status == 404:
            stats["http_404"] += 1
        elif status == 429:
            stats["http_429"] += 1
            stats["rate_limit_events"] += 1
            crawl_warnings.append(f"Rate limit terdeteksi pada {url}; crawler tidak melakukan aggressive retry.")
        elif 500 <= status < 600:
            stats["http_5xx"] += 1
        return None
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
        stats["requests_failed"] += 1
        request_log.append({"url": url, "purpose": purpose, "status": type(exc).__name__, "time": now_indonesia_str()})
        return None
    except requests.exceptions.RequestException as exc:
        stats["requests_failed"] += 1
        request_log.append({"url": url, "purpose": purpose, "status": type(exc).__name__, "time": now_indonesia_str()})
        return None


In [ ]:
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    boilerplates = [
        r"Baca juga:.*?(?= [A-Z][a-z]+|$)",
        r"Pewarta:.*$",
        r"Editor:.*$",
        r"Copyright .*?ANTARA.*$",
        r"Dilarang mengambil.*$",
    ]
    for pattern in boilerplates:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_canonical_url(soup, fallback_url):
    canonical = soup.find("link", rel=lambda value: value and "canonical" in value.lower())
    if canonical and canonical.get("href"):
        normalized = normalize_url(canonical.get("href"))
        if normalized and is_antaranews_domain(normalized):
            return normalized
    return normalize_url(fallback_url)

def extract_title(soup):
    selectors = ["h1", "article h1", ".post-title", ".article-title"]
    for selector in selectors:
        node = soup.select_one(selector)
        if node:
            title = clean_text(node.get_text(" ", strip=True))
            if title:
                return title
    og = soup.find("meta", property="og:title")
    if og and og.get("content"):
        return clean_text(og["content"])
    if soup.title:
        return clean_text(soup.title.get_text(" ", strip=True))
    return ""

def extract_publication_date(soup):
    candidates = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "pubdate"}),
        ("meta", {"name": "publishdate"}),
        ("time", {}),
    ]
    for tag, attrs in candidates:
        node = soup.find(tag, attrs=attrs) if attrs else soup.find(tag)
        if node:
            value = node.get("content") or node.get("datetime") or node.get_text(" ", strip=True)
            if value:
                return clean_text(value)
    return ""

def extract_category(soup):
    for selector in [".breadcrumb a", "nav.breadcrumb a", "a[href*='/kategori/']"]:
        nodes = soup.select(selector)
        if nodes:
            values = [clean_text(n.get_text(" ", strip=True)) for n in nodes if clean_text(n.get_text(" ", strip=True))]
            if values:
                return " > ".join(values[:4])
    return ""

def extract_author(soup):
    for selector in [".author", ".post-author", "span[itemprop='author']"]:
        node = soup.select_one(selector)
        if node:
            author = clean_text(node.get_text(" ", strip=True))
            if author:
                return author
    meta = soup.find("meta", attrs={"name": "author"})
    if meta and meta.get("content"):
        return clean_text(meta["content"])
    return ""

def extract_content(soup):
    for unwanted in soup(["script", "style", "noscript", "iframe", "form", "nav", "footer", "header", "aside"]):
        unwanted.decompose()
    selectors = [
        "div.post-content",
        "div.article-content",
        "div.entry-content",
        "div[itemprop='articleBody']",
        "article .content",
        "article",
    ]
    best_text = ""
    for selector in selectors:
        node = soup.select_one(selector)
        if not node:
            continue
        for extra in node.select(".related, .share, .social, .ads, .advertisement, .breadcrumb, .tag, .tags, .comment"):
            extra.decompose()
        paragraphs = [p.get_text(" ", strip=True) for p in node.find_all(["p", "li"])]
        text = clean_text(" ".join(paragraphs) if paragraphs else node.get_text(" ", strip=True))
        if len(text) > len(best_text):
            best_text = text
    return best_text


In [ ]:
def extract_article_urls_from_listing(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    # Batasi link pada kolom daftar artikel utama, bukan sidebar/populer.
    listing_root = soup.select_one("section .col-md-8") or soup
    found = []
    for a in listing_root.select("a[href]"):
        normalized = normalize_url(a.get("href"), base=base_url)
        if not normalized:
            continue
        if is_antaranews_domain(normalized) and looks_like_article_url(normalized) and can_fetch_url(normalized):
            found.append(normalized)
    return found

def extract_next_page_urls(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    next_urls = []
    for a in soup.find_all("a", href=True):
        text = clean_text(a.get_text(" ", strip=True)).lower()
        rel = " ".join(a.get("rel", [])).lower() if a.get("rel") else ""
        href = normalize_url(a.get("href"), base=base_url)
        if not href or not is_antaranews_domain(href) or not can_fetch_url(href):
            continue
        same_tag_or_page = "/tag/" in urlparse(href).path or urlparse(href).query
        if same_tag_or_page and (text in {"next", "berikutnya", ">", "›"} or "next" in rel or "page=" in href):
            next_urls.append(href)
    return next_urls

def build_paginated_seed(seed_url, page_number):
    normalized = normalize_url(seed_url)
    if page_number <= 1:
        return normalized
    parsed = urlparse(normalized)
    path = parsed.path.rstrip("/")
    # Pagination ANTARA menggunakan /tag/<slug>/<nomor-halaman>.
    return urlunparse((parsed.scheme, parsed.netloc, f"{path}/{page_number}", "", "", ""))

def discover_article_urls(seed_urls, max_pages=MAX_PAGES, max_candidates=MAX_ARTICLE_CANDIDATES):
    queue = deque()
    visited_pages = set()
    article_urls = []
    article_seen = set()

    for seed in seed_urls:
        normalized_seed = normalize_url(seed)
        if normalized_seed and is_antaranews_domain(normalized_seed) and can_fetch_url(normalized_seed):
            queue.append(normalized_seed)

    while queue and len(visited_pages) < max_pages and len(article_urls) < max_candidates:
        page_url = queue.popleft()
        if page_url in visited_pages:
            continue
        visited_pages.add(page_url)
        response = polite_get(page_url, purpose="listing")
        if response is None:
            continue
        found = extract_article_urls_from_listing(response.text, page_url)
        stats["total_urls_found_raw"] += len(found)
        for url in found:
            if url not in article_seen:
                article_seen.add(url)
                article_urls.append(url)
        for next_url in extract_next_page_urls(response.text, page_url):
            if next_url not in visited_pages and len(visited_pages) + len(queue) < max_pages:
                queue.append(next_url)
        # Jika pagination tidak memiliki link next, buat halaman berikutnya dari URL tag.
        parsed_page = urlparse(page_url)
        if parsed_page.path.startswith("/tag/"):
            path_parts = parsed_page.path.rstrip("/").split("/")
            current_page = int(path_parts[-1]) if path_parts[-1].isdigit() else 1
            base_seed = urlunparse((parsed_page.scheme, parsed_page.netloc, "/".join(path_parts[:-1]) if path_parts[-1].isdigit() else parsed_page.path, "", "", ""))
            generated_next = build_paginated_seed(base_seed, current_page + 1)
            if generated_next not in visited_pages and len(visited_pages) + len(queue) < max_pages:
                queue.append(generated_next)
        if len(visited_pages) % 10 == 0 or len(article_urls) >= max_candidates:
            print(f"Pages crawled: {len(visited_pages)} | Unique article URLs found: {len(article_urls)}")
        if stats["http_429"] > 0:
            crawl_warnings.append("Discovery dihentikan karena HTTP 429/rate limiting.")
            break

    stats["total_unique_article_urls"] = len(article_urls)
    return article_urls, visited_pages


In [ ]:
def count_term_hits(text, terms):
    text = clean_text(text).lower()
    hits = {}
    for term in terms:
        pattern = rf"(?<!\w){re.escape(term)}(?!\w)"
        count = len(re.findall(pattern, text))
        if count:
            hits[term] = count
    return hits

def evaluate_music_relevance(title, content):
    title_text = clean_text(title).lower()
    content_text = clean_text(content).lower()
    lead_text = content_text[:700]
    title_hits = count_term_hits(title_text, MUSIC_TERMS)
    lead_hits = count_term_hits(lead_text, MUSIC_TERMS)
    body_hits = count_term_hits(content_text, MUSIC_TERMS)
    incidental_hits = count_term_hits(title_text, INCIDENTAL_SUBJECT_TERMS)
    strong_title_hits = set(title_hits) & STRONG_MUSIC_TERMS
    strong_lead_hits = set(lead_hits) & STRONG_MUSIC_TERMS
    strong_body_hits = set(body_hits) & STRONG_MUSIC_TERMS
    music_occurrences = sum(body_hits.values())
    word_count = max(len(content_text.split()), 1)
    music_density = music_occurrences / word_count

    # Judul dan lead diberi bobot lebih besar agar penyebutan musik di bagian
    # akhir artikel umum tidak cukup untuk meloloskan artikel tersebut.
    subject_score = (3 * len(strong_title_hits)) + (2 * len(strong_lead_hits)) + min(len(strong_body_hits), 4)
    title_is_incidental = bool(incidental_hits) and len(incidental_hits) >= max(len(strong_title_hits), 1)
    sufficiently_music_focused = (
        len(strong_title_hits) >= 1 and (len(strong_lead_hits) >= 1 or len(strong_body_hits) >= 2)
    ) or (
        len(strong_lead_hits) >= 2 and len(strong_body_hits) >= 3 and music_density >= 0.005
    )
    relevant = subject_score >= RELEVANCE_MIN_MUSIC_HITS and sufficiently_music_focused and not title_is_incidental

    if relevant:
        reason = f"relevant: title_hits={sorted(strong_title_hits)}, lead_hits={sorted(strong_lead_hits)}, body_terms={len(strong_body_hits)}"
    elif title_is_incidental:
        reason = f"non-music subject in title ({sorted(incidental_hits)}); music terms appear incidentally"
    elif not sufficiently_music_focused:
        reason = f"music focus too weak: title={sorted(strong_title_hits)}, lead={sorted(strong_lead_hits)}, body_terms={len(strong_body_hits)}"
    else:
        reason = f"music subject score below threshold: {subject_score}"
    details = {"title_hits": sorted(strong_title_hits), "lead_hits": sorted(strong_lead_hits), "body_hits": sorted(strong_body_hits), "music_density": round(music_density, 4), "reason": reason}
    return relevant, reason, details

def is_relevant_music_article(title, content):
    return evaluate_music_relevance(title, content)[0]

def extract_article(url):
    if not url or not is_antaranews_domain(url) or not looks_like_article_url(url):
        stats["articles_failed"] += 1
        return None
    response = polite_get(url, purpose="article")
    if response is None:
        stats["articles_failed"] += 1
        return None
    try:
        soup = BeautifulSoup(response.text, "html.parser")
        canonical_url = get_canonical_url(soup, url)
        title = extract_title(soup)
        content = extract_content(soup)
        if not content:
            stats["empty_content"] += 1
        if not title or not content:
            stats["articles_failed"] += 1
            return None
        stats["articles_extracted"] += 1
        is_relevant, relevance_reason, relevance_details = evaluate_music_relevance(title, content)
        if not is_relevant:
            stats["irrelevant_articles"] += 1
            if len(rejected_examples) < REJECTED_EXAMPLES_LIMIT:
                rejected_examples.append({"title": title, "url": canonical_url, "reason": relevance_reason})
            return None
        stats["relevant_articles"] += 1
        return {
            "title": title,
            "url": canonical_url,
            "content": content,
            "crawl_date": now_indonesia_str(),
            "publication_date": extract_publication_date(soup),
            "category": extract_category(soup),
            "author": extract_author(soup),
            "source_tag": "Musik Indonesia",
        }
    except Exception as exc:
        stats["articles_failed"] += 1
        request_log.append({"url": url, "purpose": "parse_article", "status": type(exc).__name__, "time": now_indonesia_str()})
        return None

def crawl_articles(article_urls, target_documents=TARGET_DOCUMENTS):
    records = []
    processed = 0
    for url in article_urls[:MAX_ARTICLE_CANDIDATES]:
        if len(records) >= target_documents:
            break
        if stats["http_429"] > 0:
            crawl_warnings.append("Article crawling dihentikan karena HTTP 429/rate limiting.")
            break
        processed += 1
        record = extract_article(url)
        if record:
            records.append(record)
        if processed % 25 == 0 or processed == len(article_urls):
            print(f"[{processed}/{len(article_urls)}] candidates processed | relevant: {len(records)}")
    return records


## 6. URL Discovery

Tahap ini mengumpulkan kandidat URL artikel dari halaman tag dan pagination. URL disaring agar tetap berada pada domain ANTARA, terlihat seperti URL artikel, tidak berada pada path yang dikecualikan, dan diizinkan oleh robots.txt.

In [ ]:
article_urls, visited_listing_pages = discover_article_urls(SEED_URLS)
print("Pages crawled:", len(visited_listing_pages))
print("Unique article URLs found:", len(article_urls))
article_urls[:10]

## 7. Article Crawling dan Relevance Filter

Setelah title dan isi utama berhasil diekstrak, artikel dinilai dengan filter relevansi deterministik. Filter memprioritaskan istilah musik pada judul dan lead, memeriksa cakupan istilah pada isi, menghitung kepadatan istilah musik, dan menolak judul yang jelas berfokus pada otomotif, teknologi, politik, pariwisata, atau topik umum lain. Isi artikel yang ditolak tidak dimasukkan ke dataset final.

In [ ]:
raw_records = crawl_articles(article_urls, TARGET_DOCUMENTS)
print("Total extracted records before cleaning/deduplication:", len(raw_records))

[25/2014] candidates processed | relevant: 16
[50/2014] candidates processed | relevant: 32
[75/2014] candidates processed | relevant: 48
[100/2014] candidates processed | relevant: 66
[125/2014] candidates processed | relevant: 78
[150/2014] candidates processed | relevant: 96
[175/2014] candidates processed | relevant: 105
[200/2014] candidates processed | relevant: 120
[225/2014] candidates processed | relevant: 132
[250/2014] candidates processed | relevant: 150
[275/2014] candidates processed | relevant: 174
[300/2014] candidates processed | relevant: 184
[325/2014] candidates processed | relevant: 205
[350/2014] candidates processed | relevant: 216
[375/2014] candidates processed | relevant: 235
[400/2014] candidates processed | relevant: 251
[425/2014] candidates processed | relevant: 271
[450/2014] candidates processed | relevant: 286
[475/2014] candidates processed | relevant: 301
[500/2014] candidates processed | relevant: 320
[525/2014] candidates processed | relevant: 332
[

## 8. Cleaning dan Deduplication

Cleaning dilakukan untuk menghapus whitespace berlebihan, boilerplate umum, content kosong, artikel terlalu pendek, title kosong, dan URL kosong. Stemming dan stopword removal tidak dilakukan karena dataset ini akan digunakan kembali untuk tahap preprocessing pada Tugas 2.

In [ ]:
def content_hash(title, content):
    key = clean_text((title or "") + " " + (content or "")).lower()
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def clean_and_deduplicate(records):
    cleaned = []
    seen_urls = set()
    seen_hashes = set()
    before = len(records)

    for record in records:
        title = clean_text(record.get("title", ""))
        url = normalize_url(record.get("url", ""))
        content = clean_text(record.get("content", ""))

        if not title or not url or not content or len(content) < MIN_CONTENT_LENGTH:
            stats["discarded_cleaning"] += 1
            continue
        if url in seen_urls:
            stats["duplicates"] += 1
            continue
        c_hash = content_hash(title, content)
        if c_hash in seen_hashes:
            stats["duplicates"] += 1
            continue

        seen_urls.add(url)
        seen_hashes.add(c_hash)
        new_record = dict(record)
        new_record["title"] = title
        new_record["url"] = url
        new_record["content"] = content
        cleaned.append(new_record)

    df = pd.DataFrame(cleaned)
    if not df.empty:
        df = df.reset_index(drop=True)
        df.insert(0, "doc_id", [f"D{i:03d}" for i in range(1, len(df) + 1)])
        required_first = ["doc_id", "title", "url", "content", "crawl_date"]
        remaining = [col for col in df.columns if col not in required_first]
        df = df[required_first + remaining]
    else:
        df = pd.DataFrame(columns=["doc_id", "title", "url", "content", "crawl_date"])

    cleaning_summary = {
        "records_before_cleaning": before,
        "discarded_cleaning": stats["discarded_cleaning"],
        "duplicates": stats["duplicates"],
        "final_documents": len(df),
    }
    return df, cleaning_summary

df, cleaning_summary = clean_and_deduplicate(raw_records)
cleaning_summary

{'records_before_cleaning': 1000,
 'discarded_cleaning': 3,
 'duplicates': 0,
 'final_documents': 997}

## 9. Dataset Preview


In [ ]:
df.head()

,doc_id,title,url,content,crawl_date,publication_date,category,author,source_tag
0,D001,"Gloria Jessica dan Ade Govinda rilis album ""Bl...",https://www.antaranews.com/berita/5768940/glor...,Jakarta (ANTARA) - Musisi Gloria Jessica dan A...,2026-10-08 16:09:07,2026-10-02T19:11:00+07:00,,antaranews.com,Musik Indonesia
1,D002,"700 delegasi musisi akan hadiri kegiatan ""KMI ...",https://www.antaranews.com/berita/5765815/700-...,Makassar (ANTARA) - Pemerintah Kota Makassar t...,2026-10-08 16:09:09,2026-10-01T08:14:59+07:00,,antaranews.com,Musik Indonesia
2,D003,Reality Club dan Phum Viphurit akan tampil di ...,https://www.antaranews.com/berita/5765643/real...,Jakarta (ANTARA) - Synchronize Festival 2026 a...,2026-10-08 16:09:13,2026-09-30T23:47:45+07:00,,antaranews.com,Musik Indonesia
3,D004,Project Pop satukan generasi lewat lagu di Pes...,https://www.antaranews.com/berita/5760947/proj...,Jakarta (ANTARA) - Grup musik Project Pop meng...,2026-10-08 16:09:15,2026-09-27T23:53:21+07:00,,antaranews.com,Musik Indonesia
4,D005,Ndarboy dan Happy Asmara bawa musik Jawa dalam...,https://www.antaranews.com/berita/5760409/ndar...,Jakarta (ANTARA) - Penyanyi Ndarboy dan Happy ...,2026-10-08 16:09:19,2026-09-27T12:36:12+07:00,,antaranews.com,Musik Indonesia


In [ ]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 997 entries, 0 to 996
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   doc_id            997 non-null    object
 1   title             997 non-null    object
 2   url               997 non-null    object
 3   content           997 non-null    object
 4   crawl_date        997 non-null    object
 5   publication_date  997 non-null    object
 6   category          997 non-null    object
 7   author            997 non-null    object
 8   source_tag        997 non-null    object
dtypes: object(9)
memory usage: 70.2+ KB


(997, 9)

In [ ]:
preview_cols = ["doc_id", "title", "url", "content", "crawl_date"]
preview_df = df[preview_cols].head(5).copy() if not df.empty else pd.DataFrame(columns=preview_cols)
if not preview_df.empty:
    preview_df["content"] = preview_df["content"].str.slice(0, 300) + "..."
preview_df

,doc_id,title,url,content,crawl_date
0,D001,"Gloria Jessica dan Ade Govinda rilis album ""Bl...",https://www.antaranews.com/berita/5768940/glor...,Jakarta (ANTARA) - Musisi Gloria Jessica dan A...,2026-10-08 16:09:07
1,D002,"700 delegasi musisi akan hadiri kegiatan ""KMI ...",https://www.antaranews.com/berita/5765815/700-...,Makassar (ANTARA) - Pemerintah Kota Makassar t...,2026-10-08 16:09:09
2,D003,Reality Club dan Phum Viphurit akan tampil di ...,https://www.antaranews.com/berita/5765643/real...,Jakarta (ANTARA) - Synchronize Festival 2026 a...,2026-10-08 16:09:13
3,D004,Project Pop satukan generasi lewat lagu di Pes...,https://www.antaranews.com/berita/5760947/proj...,Jakarta (ANTARA) - Grup musik Project Pop meng...,2026-10-08 16:09:15
4,D005,Ndarboy dan Happy Asmara bawa musik Jawa dalam...,https://www.antaranews.com/berita/5760409/ndar...,Jakarta (ANTARA) - Penyanyi Ndarboy dan Happy ...,2026-10-08 16:09:19


## 10. Dataset Quality Check

Quality check memastikan tidak ada nilai kosong pada kolom wajib, tidak ada duplikasi URL, tidak ada duplikasi `doc_id`, dan tidak ada content kosong atau terlalu pendek.

In [ ]:
quality_report = {
    "missing_values": df.isna().sum().to_dict(),
    "duplicated_url": int(df["url"].duplicated().sum()) if "url" in df else 0,
    "duplicated_doc_id": int(df["doc_id"].duplicated().sum()) if "doc_id" in df else 0,
    "empty_content": int((df["content"].str.len() == 0).sum()) if not df.empty else 0,
    "too_short_content": int((df["content"].str.len() < MIN_CONTENT_LENGTH).sum()) if not df.empty else 0,
}
quality_report

{'missing_values': {'doc_id': 0,
  'title': 0,
  'url': 0,
  'content': 0,
  'crawl_date': 0,
  'publication_date': 0,
  'category': 0,
  'author': 0,
  'source_tag': 0},
 'duplicated_url': 0,
 'duplicated_doc_id': 0,
 'empty_content': 0,
 'too_short_content': 0}

## 11. Statistik Crawling

Statistik berikut dihitung dari proses eksekusi aktual. Angka tidak di-hard-code.

In [ ]:
stats["final_documents"] = len(df)
stats["documents_after_cleaning"] = len(df)
filtering_stats = {
    "Candidate URLs found": stats["total_urls_found_raw"],
    "Unique candidate URLs": stats["total_unique_article_urls"],
    "Articles successfully extracted": stats["articles_extracted"],
    "Relevant articles": stats["relevant_articles"],
    "Irrelevant articles": stats["irrelevant_articles"],
    "Failed/empty articles": stats["articles_failed"] + stats["empty_content"],
    "Duplicate articles": stats["duplicates"],
    "Final documents": stats["final_documents"],
}
stats_df = pd.DataFrame(list(filtering_stats.items()), columns=["metric", "value"])
stats_df

,metric,value
0,Candidate URLs found,5120
1,Unique candidate URLs,2014
2,Articles successfully extracted,1603
3,Relevant articles,1000
4,Irrelevant articles,603
5,Failed/empty articles,4
6,Duplicate articles,0
7,Final documents,997


### Contoh Artikel yang Ditolak

Contoh berikut disimpan selama eksekusi dan menunjukkan judul, URL, serta alasan deterministik mengapa artikel tidak dianggap sebagai dokumen Musik Indonesia.

In [ ]:
rejected_examples_df = pd.DataFrame(rejected_examples, columns=["title", "url", "reason"])
rejected_examples_df

,title,url,reason
0,Riset Evident: Wamen dan waka badan berperan d...,https://www.antaranews.com/berita/5776848/rise...,"music focus too weak: title=[], lead=[], body_..."
1,BYD Atto 3 EX meluncur hingga arah pengembanga...,https://www.antaranews.com/berita/5765764/byd-...,"non-music subject in title (['byd', 'openai'])..."
2,Panglima TNI: Festival band jadi ruang cari ba...,https://www.antaranews.com/berita/5760744/pang...,"non-music subject in title (['panglima', 'tni'..."
3,QRIS Star didorong jadi klasifikasi produk eko...,https://www.antaranews.com/berita/5759743/qris...,"music focus too weak: title=[], lead=[], body_..."
4,Dedikasi seni dan warisan Guruh untuk kebudaya...,https://www.antaranews.com/berita/5759441/dedi...,"music focus too weak: title=[], lead=['lagu'],..."
5,"Kemarin, IDAI imbau anak vaksin hingga Hyundai...",https://www.antaranews.com/berita/5752069/kema...,"music focus too weak: title=[], lead=[], body_..."
6,Kemenbud-Kedubes Maroko perkuat diplomasi kebu...,https://www.antaranews.com/berita/5671373/keme...,"music focus too weak: title=[], lead=[], body_..."
7,Indonesia-Ethiopia perkuat diplomasi budaya di...,https://www.antaranews.com/berita/5669353/indo...,"music focus too weak: title=[], lead=['musik']..."
8,Ragam rekomendasi wisata akhir pekan di DKI Ja...,https://www.antaranews.com/berita/5655044/raga...,non-music subject in title (['wisata']); music...
9,Yonatan Pandelaki napak tilas budaya di Samosi...,https://www.antaranews.com/berita/5639641/yona...,"music focus too weak: title=[], lead=['musisi'..."


In [ ]:
if crawl_warnings:
    print("Catatan/kendala crawling:")
    for warning in crawl_warnings:
        print("-", warning)
else:
    print("Tidak ada warning khusus yang tercatat.")

Tidak ada warning khusus yang tercatat.


## 12. Export CSV

Dataset final disimpan sebagai `dataset_musik_indonesia_antaranews.csv`. File hanya berisi data hasil crawling aktual dari ANTARA, bukan data sintetis.

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Dataset disimpan ke: {OUTPUT_FILE}")
print(f"Jumlah dokumen final: {len(df)}")

Dataset disimpan ke: dataset_musik_indonesia_antara.csv
Jumlah dokumen final: 997


In [ ]:
# Jalankan cell ini di Google Colab untuk mendownload CSV.
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception as exc:
    print("Download otomatis hanya tersedia di Google Colab. File tetap tersimpan di runtime.")
    print(type(exc).__name__, exc)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>